# ML Lab · Module 4, Unit 9: Credit risk

**Prof. Dr. Mehmet Songur** · ML Lab, Module 4

Data: 30,000 credit card customers of a bank in Taiwan (UCI, Yeh and Lien 2009, CC BY 4.0). The setup cell downloads a table called `credit`. The target `default`: default in October 2005 (1). Columns: `limit_bal` (credit limit), `sex`, `education`, `marriage`, `age`, `pay_0`, `pay_2` … `pay_6` (payment status for September … April), `bill_amt1` … `bill_amt6` (statement amount), `pay_amt1` … `pay_amt6` (amount paid). **Have Gemini write the code**, then **audit it**.

**Payment status codes (`pay_0`, `pay_2` … `pay_6`):** the UCI data dictionary defines −1 as “pay duly” and 1, 2, … as “months of delay”; it gives no description for −2 and 0. **Course assumption** (the common reading in the literature): −2 card not used, −1 balance paid in full, 0 minimum payment made. `pay_amt1` … `pay_amt6`, by contrast, are amounts, not codes. Source: Yeh (2009), UCI, doi.org/10.24432/C55S3H.

**Before you start:** to edit this notebook opened from GitHub, choose **File → Save a copy in Drive** and work in the copy.

**Flow and rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running. Gemini may add its code in a new cell; run the cells in order.
3. When the task code runs without errors, run that task's **Check** cell; if it reports a problem, fix it and rerun the task and the check.
4. If a check passes it gives a **completion code**: Task A `ML4-KRD-…`, Task B `ML4-ESK-…`. Paste it and the task code into the matching box in ML Lab.

> **Scope:** this is a retrospective, single-period comparison on existing card customers in 2005; it must be tested separately for new credit applications or later months.

In [ ]:
#@title Setup — run it, do not change it

import base64, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("always", category=ConvergenceWarning)   # yakınsama uyarısı görünür kalır / convergence warnings stay visible

credit = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/credit/taiwan_default.csv")
print("credit:", credit.shape, "| default rate:", round(credit["default"].mean(), 3))
_K = credit.copy()
_E, _T = train_test_split(_K, test_size=0.3, random_state=0, stratify=_K["default"])

def _k(s):
    return base64.b64decode(s).decode()

def _cost(y, p, e):
    k = p >= e
    return (5 * np.sum(~k & (y == 1)) + np.sum(k & (y == 0))) / len(y)

def _last_step(model):
    return model.steps[-1][1] if isinstance(model, Pipeline) else model

_STATUS = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]

def _ref_prepare(df):
    X = df.drop(columns=["default", "sex", "marriage"]).copy()
    for k in _STATUS:
        X[k + "_delay"] = X[k].clip(lower=0); X[k + "_full"] = (X[k] == -1).astype(int); X[k + "_none"] = (X[k] == -2).astype(int)
        X = X.drop(columns=[k])
    for k in [c for c in X.columns if "amt" in c]:
        X[k] = np.sign(X[k]) * np.log1p(np.abs(X[k]))
    return X

def _example():
    o = _K.head(3).copy()
    o[_STATUS] = [[-2, -1, 0, 2, 1, 3], [0, 4, -2, -1, 0, 1], [1, -2, -1, 0, 5, -2]]   # every code type
    o["bill_amt1"] = [-1500, 0, 82000]; o["pay_amt1"] = [0, 2500, 17]                  # signed amounts
    return o

def _verify(prepare):
    v = pd.concat([_example(), _E.head(3000)])
    Xs, Xr = prepare(v.copy()), _ref_prepare(v)
    if len(Xs) != len(Xr):
        return "row count changed", None
    S = {c: np.asarray(Xs[c], dtype=float) for c in Xs.columns if pd.api.types.is_numeric_dtype(Xs[c])}
    matched, missing = set(), []
    for c in Xr.columns:
        r = Xr[c].to_numpy(dtype=float)
        m = next((sc for sc, sv in S.items() if sc not in matched and np.allclose(sv, r)), None)
        if m is None: missing.append(c)
        else: matched.add(m)
    extra = [c for c in Xs.columns if c not in matched]
    return missing, extra

def check_A(train, test, prepare, model, p_test, auc_test):
    if len(test) != len(_T) or set(test.index) != set(_T.index) or set(train.index) != set(_E.index):
        print("❌ The split differs from the expected one: train_test_split(credit, test_size=0.3, random_state=0, stratify=credit['default'])."); return
    Xe = prepare(train.copy())
    if {"sex", "marriage"} & set(Xe.columns):
        print("❌ Protected attributes (sex, marriage) are in the model. Leave them out."); return
    if "default" in Xe.columns:
        print("❌ The target ('default') is among the features: target leakage."); return
    raw = [k for k in _STATUS if k in Xe.columns]
    if raw:
        print(f"❌ The payment status codes are still numbers ({', '.join(raw)}). −2 and −1 are categories: months of delay, full payment and non-use should be separate variables."); return
    missing, extra = _verify(prepare)
    if extra is None:
        print("❌ prepare() changed the number of rows; no rows should be dropped or duplicated."); return
    if missing or extra:
        amts = [c for c in missing if "amt" in c]
        if any(str(c).startswith("pay_amt") and len(str(c)) > 8 for c in extra):
            print("❌ The pay_amt1–6 amounts appear to have been treated as status codes (every column starting with pay_ may have been selected). Decode only pay_0, pay_2 … pay_6; keep the pay_amt amounts and apply the signed log.")
        elif amts:
            print(f"❌ These amounts do not match the expected signed log transform: {', '.join(amts[:6])}{' …' if len(amts) > 6 else ''}. The transform should be sign(x)·log(1+|x|).")
        else:
            print(f"❌ The features do not match the expected 33 columns. Missing or computed differently: {', '.join(missing[:6]) or '—'}; extra: {', '.join(map(str, extra[:6])) or '—'}. Delay = clip the value below at 0; full = (value == −1); none = (value == −2).")
        return
    if not isinstance(model, Pipeline) or not any(isinstance(s, StandardScaler) for _, s in model.steps):
        print("❌ Scaling and logistic regression are not in a pipeline (make_pipeline(StandardScaler(), LogisticRegression(...)))."); return
    last = _last_step(model)
    if not isinstance(last, LogisticRegression):
        print("❌ The pipeline's last step should be LogisticRegression."); return
    if last.class_weight is not None:
        print("❌ class_weight was used: the probabilities inflate and are no longer default probabilities. Remove class_weight."); return
    if last.C < 1e4:
        print("ℹ️ Info: C is small; the logistic regression is clearly penalised. The request was C=1e6 (a very small penalty).")
    p = np.asarray(p_test, dtype=float)
    if len(p) != len(test) or len(np.unique(p)) <= 2:
        print("❌ p_test is not a probability (only 0/1, or the wrong length). Use predict_proba(...)[:, 1] instead of predict."); return
    if list(last.classes_) != [0, 1] or not np.allclose(p, model.predict_proba(prepare(test.copy()))[:, 1]):
        print("❌ p_test does not match the model's default (1) probabilities for prepare(test) (predict_proba(...)[:, 1])."); return
    a = roc_auc_score(test["default"], p)
    if abs(float(auc_test) - a) > 1e-6:
        print("❌ auc_test does not match roc_auc_score(test['default'], p_test)."); return
    if a < 0.75:
        print(f"❌ Test AUC {a:.3f}: lower than expected (≈ 0.76)."); return
    print("Transformations verified (33 features, including example rows with −2, −1, 0, positive delays and signed amounts): 6 delay, 6 full-payment, 6 non-use indicators, 12 signed-log amounts, 3 others (limit_bal, education, age).")
    g = pd.qcut(p, 10, labels=range(1, 11))
    k = pd.DataFrame({"p": p, "y": test["default"].values}).groupby(g, observed=True).agg(n=("y", "size"), mean_probability=("p", "mean"), realised=("y", "mean"))
    print("Calibration table for comparison (yours should match):"); print(k.round(3).to_string())
    print(f"✅ Task A checks passed (test AUC ≈ {a:.3f}). Your Task A completion code: " + _k("TUw0LUtSRC04TldU"))

def check_B(train, test, prepare, model, p_test, thr, cost_test):
    grid = np.round(np.arange(0.05, 0.71, 0.01), 2)
    ye, yt = train["default"].values, test["default"].values
    p_cv = cross_val_predict(clone(model), prepare(train), ye, cv=5, method="predict_proba")[:, 1]
    ref = grid[np.argmin([_cost(ye, p_cv, e) for e in grid])]
    tst = grid[np.argmin([_cost(yt, np.asarray(p_test), e) for e in grid])]
    e = float(thr)
    if abs(e - ref) > 0.006:
        if abs(e - tst) < 0.006 and abs(tst - ref) > 0.006:
            print(f"❌ The threshold was chosen on the test data ({e:.2f}). Choose it on the training data with cross_val_predict probabilities."); return
        print(f"❌ The threshold ({e:.2f}) differs from the expected one. Costs: missed default 5, rejected good customer 1; grid 0.05–0.70; cross_val_predict with cv=5 (no shuffling) on training."); return
    m = _cost(yt, np.asarray(p_test), e)
    if abs(float(cost_test) - m) > 0.002:
        print("❌ cost_test differs from the expected value: it should be the average cost per application (5·FN + FP) / n on the test data at the chosen threshold."); return
    pt = np.asarray(p_test)
    print(f"Threshold chosen on training from the grid (step 0.01) {e:.2f}: test cost {m:.4f}")
    print(f"Exact 1/6 = 0.1667 set in advance: test cost {_cost(yt, pt, 1 / 6):.4f}  ·  threshold 0.5: {_cost(yt, pt, 0.5):.4f}  ·  approve everyone: {_cost(yt, pt, 2):.4f}")
    print("Note: a small gap on a single held-out test is not evidence that one threshold is better in general.")
    print("✅ Task B checks passed. Your Task B completion code: " + _k("TUw0LUVTSy0zSEpW"))

credit.head(3)

## Task A — Model

1. Split `credit` into `train` and `test` with `train_test_split(credit, test_size=0.3, random_state=0, stratify=credit["default"])`.
2. Write a function `prepare(df)`: drop the `default`, `sex` and `marriage` columns; **only** for each status code `pay_0`, `pay_2` … `pay_6` create `_delay` (clipped below at 0), `_full` (1 if the value is −1) and `_none` (1 if the value is −2) and drop the original status column (`pay_amt1–6` are amounts; leave them alone); transform the 12 amounts whose names contain `amt` (`bill_amt1–6`, `pay_amt1–6`) with a signed log: `sign(x)·log(1+|x|)`. The result is 33 features.
3. Fit `make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000))` on `prepare(train)` only: `model`.
4. Test probabilities `p_test = model.predict_proba(prepare(test))[:, 1]` (second column: `model.classes_` = [0, 1]), test AUC `auc_test`.
5. **Calibration table:** split `p_test` into ten groups with `pd.qcut`; print each group's count, mean probability and realised default rate.

**Audit questions:** Are sex and marriage left out? Are only the status codes decoded (not the `pay_amt` amounts)? Are there 33 features? Is the scaler inside the pipeline? Was AUC computed with `predict_proba`? Is there any `class_weight`?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(train, test, prepare, model, p_test, auc_test)

## Task B — A cost-based threshold

A missed default (FN) costs 5 units, a rejected good customer (FP) 1. Average cost per application: (5·FN + FP) / n.

1. On the training data, produce cross-validation probabilities with `cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000)), prepare(train), train["default"], cv=5, method="predict_proba")[:, 1]`.
2. Compute the training cost on a threshold grid from 0.05 to 0.70 in steps of 0.01; choose the lowest as `thr`.
3. On the test data, compute the cost at the chosen threshold: `cost_test`.
4. Compare on separate rows: “approve everyone”, threshold 0.5, the grid threshold and the exact 1/6 set in advance (≈ 0.1667; not the same as 0.17).

**Audit questions:** Was the threshold chosen with the test data? Is the small cost gap between the grid threshold and 1/6 a general advantage, or the noise of a single test?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it (it may take a few seconds)

In [ ]:
check_B(train, test, prepare, model, p_test, thr, cost_test)

## Your audit notes

- How did Gemini prepare the features? Did you need to change anything?
- How were scaling, AUC and the threshold choice done?
- How would you explain the result to a credit committee? (AUC, grouped calibration, threshold, cost; limitations, scope and fairness)